# 48 · Spark ML feature engineering and regression

**Scenario:** Train a bounded fare baseline using only Databricks taxi asset records.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **spark-ml**. Requires Spark ML support on the attached recent Databricks environment. No sklearn or external training dataset is used.

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Split before fitting preprocessing

Distance and pickup hour predict fare in this mechanics lab. Real-time model features must be available at prediction time; evaluate that contract before use.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
usable=(df.filter('distance > 0 AND fare >= 0 AND pickup_ts IS NOT NULL')
    .withColumn('pickup_hour',F.hour('pickup_ts').cast('double')).withColumn('label',F.col('fare')))
train,test=usable.randomSplit([0.8,0.2],42)
assert train.count()>10 and test.count()>0, 'Increase max_rows for enough valid train/test rows.'
pipeline=Pipeline(stages=[VectorAssembler(inputCols=['distance','pickup_hour'],outputCol='raw_features'),
    StandardScaler(inputCol='raw_features',outputCol='features'),LinearRegression(regParam=0.1,maxIter=30)])
model=pipeline.fit(train)


**Expected check:** The scaler and regression fit only training rows.


## Step 3 — Evaluate against a train-mean baseline

A valid metric does not guarantee useful prediction. Fare depends on many omitted factors; a random split also differs from a future-time evaluation.


In [ ]:
scored=model.transform(test)
evaluator=RegressionEvaluator(labelCol='label',predictionCol='prediction',metricName='rmse')
rmse=evaluator.evaluate(scored)
mean_fare=train.agg(F.avg('label')).first()[0]
baseline=evaluator.evaluate(test.withColumn('prediction',F.lit(mean_fare)))
assert rmse>=0 and baseline>=0
print({'model_rmse':rmse,'train_mean_baseline_rmse':baseline})
scored.select('distance','pickup_hour','label','prediction').show(10)


**Expected check:** Both model and baseline metrics are reported; improvement is measured rather than assumed.


## Practice

Replace the random split with an asset-time cutoff and log features/metrics/model with MLflow.

## Solution guide

Fit all preprocessing before the cutoff, evaluate later records, and log the source asset, sample definition, environment, split and signature. A content fingerprint is not a feature.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
